# Reachy gripper force sensor test

Checks whether the gripper force sensors respond to force, and on which side. Run the cells top to bottom.

1. **Baseline**: grippers empty, nobody touching. Measures the offset and noise, and how often the readings are corrupted.
2. **Hand press**: the notebook prompts you when to press each gripper (timed windows), so it can tell which sensor reacts to which side.
3. **Motor squeeze** (optional): the gripper closes on an object by itself. This test works even if the grippers don't move by hand.
4. **Plot and CSV.**

**Corrupted samples.** The first run showed values like `-1`, `-16385`, `-524289` and `-655361`. In binary these are all ones except one or two bits: bad reads from the load-cell ADC, not force. They are flagged as *glitches* and left out of every statistic. The glitch rate is reported per sensor, because a high rate points to a wiring, power or board problem.

In [ ]:
# ---- Settings ----
HOST = 'localhost'
RATE = 20                 # samples per second
BASELINE_S = 5            # step 1: grippers empty, hands off
PRESS_S, REST_S = 5, 4    # step 2: length of each press window and of the rest windows between them
SQUEEZE_SIDES = ['right', 'left']   # step 3: grippers to close by motor; [] skips the step
OUTPUT_CSV = 'force_log.csv'

# Gripper angles (deg) for the left gripper, from reachy_grasping.ipynb; the right gripper turns the other way.
GRIPPER_OPEN_L, GRIPPER_CLOSED_L = 35, -12
CLOSE_S, HOLD_S = 6, 3    # step 3: time to close slowly, then time to hold closed
RESPONSE_Z = 8            # a shift of this many noise-widths (robust std) counts as a response

In [ ]:
import csv
import math
import time

import numpy as np

LOG = []        # rows: [t, phase, *raw sensor values, l_gripper_deg, r_gripper_deg, l_goal_deg, r_goal_deg]
T0 = None


def as_float(v):
    try:
        f = float(v)
        return None if math.isnan(f) else f
    except (TypeError, ValueError):
        return None


def is_glitch(v):
    # True for a corrupted ADC read: None/NaN/error, or a negative value whose bitwise NOT has <= 2 bits set
    # in the 24-bit word (-1, -16385 = ~0x4000, -524289 = ~0x80000, -655361 = ~0xA0000, ...).
    f = as_float(v)
    if f is None:
        return True
    i = int(round(f))
    return i < 0 and bin((~i) & 0xFFFFFF).count('1') <= 2


def clean(values):
    # Numeric values with glitches removed.
    return np.array([as_float(v) for v in values if not is_glitch(v)], dtype=float)


def robust_stats(arr):
    # (median, robust std from the MAD; the std is never below 1 raw unit).
    if len(arr) == 0:
        return float('nan'), float('nan')
    med = float(np.median(arr))
    return med, max(1.4826 * float(np.median(np.abs(arr - med))), 1.0)


def find_force_sensors(reachy):
    # Returns {name: sensor}. Tries the documented container first, then searches the arms.
    sensors = {}
    container = getattr(reachy, 'force_sensors', None)
    if container is not None:
        items = container.values() if hasattr(container, 'values') else \
            [v for k, v in vars(container).items() if not k.startswith('_')]
        for s in items:
            sensors[getattr(s, 'name', repr(s))] = s
    for arm_name in ('l_arm', 'r_arm'):
        arm = getattr(reachy, arm_name, None)
        if arm is None:
            continue
        for attr in dir(arm):
            if 'force' in attr.lower() and not attr.startswith('_'):
                s = getattr(arm, attr)
                if hasattr(s, 'force'):
                    sensors.setdefault(getattr(s, 'name', f'{arm_name}.{attr}'), s)
    return sensors


def read(sensor):
    try:
        return sensor.force
    except Exception as e:  # an exception here is itself a finding; it is logged and counted as a glitch
        return f'ERROR {type(e).__name__}: {e}'


def gripper_of(reachy, side):
    p = side[0]
    arm = getattr(reachy, f'{p}_arm', None)
    return None if arm is None else getattr(arm, f'{p}_gripper', None)


def _pos(g, attr):
    v = getattr(g, attr, None) if g is not None else None
    return round(v, 2) if isinstance(v, (int, float)) else None


def sample(phase):
    vals = [read(SENSORS[n]) for n in NAMES]
    gs = [GRIPPERS['left'], GRIPPERS['right']]
    LOG.append([round(time.time() - T0, 3), phase, *vals,
                *[_pos(g, 'present_position') for g in gs], *[_pos(g, 'goal_position') for g in gs]])
    return vals


def record(phase, seconds, banner=None, each=None):
    # Logs every sensor for `seconds` under `phase`; `each(tau)` is called every tick (tau goes 0 -> 1).
    if banner:
        print(banner)
    ts = time.time()
    while True:
        tau = (time.time() - ts) / seconds
        if tau >= 1:
            break
        if each:
            each(tau)
        vals = sample(phase)
        left = seconds - (time.time() - ts)
        shown = '  '.join(f'{n}: {"GLITCH" if is_glitch(v) else f"{as_float(v):10.0f}"}' for n, v in zip(NAMES, vals))
        print(f'\r   {phase:18s} {left:4.1f}s left   {shown}   ', end='', flush=True)
        time.sleep(1 / RATE)
    print()


def phase_values(name, phases, skip_s=0.0):
    # Raw values of sensor `name` logged in any of `phases`, skipping the first `skip_s` of each phase (reaction time).
    i = 2 + NAMES.index(name)
    start, out = {}, []
    for r in LOG:
        if r[1] in phases:
            start.setdefault(r[1], r[0])
            if r[0] - start[r[1]] >= skip_s:
                out.append(r[i])
    return out


def glitch_report(name, raw):
    n_bad = sum(is_glitch(v) for v in raw)
    c = clean(raw)
    changes = int(np.count_nonzero(np.diff(c))) if len(c) > 1 else 0
    return n_bad, len(raw), c, changes

## Connect

In [ ]:
from reachy_sdk import ReachySDK

print(f'connecting to {HOST} ...')
reachy = ReachySDK(host=HOST)
time.sleep(1)  # let the first sensor messages arrive

SENSORS = find_force_sensors(reachy)
assert SENSORS, f'No force sensors found. reachy attributes: {[a for a in dir(reachy) if not a.startswith("_")]}'
NAMES = list(SENSORS)
GRIPPERS = {side: gripper_of(reachy, side) for side in ('left', 'right')}
T0 = time.time()
LOG.clear()

print(f'found {len(SENSORS)} force sensor(s):')
for n, s in SENSORS.items():
    print(f'  {n:20s} {type(s).__name__:14s} force = {read(s)!r}')
for side, g in GRIPPERS.items():
    print(f'  {side:5s} gripper: ' + ('not found' if g is None else
          f'{g.name}  present {g.present_position:.1f} deg  compliant={g.compliant}'))

## 1. Baseline

Leave both grippers empty and don't touch the robot.

In [ ]:
record('baseline', BASELINE_S, banner=f'Baseline: hands off for {BASELINE_S} s ...')

BASE = {}
for n in NAMES:
    raw = phase_values(n, ['baseline'])
    n_bad, n_all, c, changes = glitch_report(n, raw)
    med, noise = robust_stats(c)
    BASE[n] = (med, noise)
    print(f'\n{n}:')
    print(f'  glitches {n_bad}/{n_all} ({100 * n_bad / max(n_all, 1):.0f}%)   clean median {med:.0f}   '
          f'noise (robust std) {noise:.0f}   clean value changed {changes} times (~{changes / BASELINE_S:.1f} updates/s)')
    if len(c) == 0:
        print('  -> no valid readings at all: the sensor is declared but no real data comes through.')
    elif changes == 0:
        print('  -> never changes: stale value, the sensor or its board is not publishing.')
    if n_bad / max(n_all, 1) > 0.05:
        print('  -> many corrupted reads: check this sensor\'s cable and connector, and the board power.')

## 2. Hand press

The cell prints **REST** or **PRESS** with a countdown. During a PRESS window, squeeze that gripper's two fingers together
(or press on the finger with the sensor) firmly, and hold until the window ends. During REST, let go completely.
It doesn't matter if the gripper doesn't move: the load cell measures force, not position.

In [ ]:
steps = [('rest_1', REST_S, 'REST: hands off'),
         ('press_left', PRESS_S, '>>> PRESS the LEFT gripper (robot\'s left) and hold'),
         ('rest_2', REST_S, 'REST: let go'),
         ('press_right', PRESS_S, '>>> PRESS the RIGHT gripper (robot\'s right) and hold'),
         ('rest_3', REST_S, 'REST: let go')]
print('Get ready: the first window starts in 3 s.')
time.sleep(3)
for phase, secs, banner in steps:
    record(phase, secs, banner=banner)

print('\nShift from the rest level, in noise-widths (z). |z| >= %d counts as a response.' % RESPONSE_Z)
print(f'{"sensor":20s} {"rest median":>12s} {"noise":>7s} {"press LEFT":>18s} {"press RIGHT":>18s}   glitches')
HAND = {}
for n in NAMES:
    rest = clean(phase_values(n, ['rest_1', 'rest_2', 'rest_3'], skip_s=1.0))
    med, noise = robust_stats(rest)
    noise = max(noise, BASE[n][1])
    cols = []
    for p in ('press_left', 'press_right'):
        c = clean(phase_values(n, [p], skip_s=1.0))
        delta = float(np.median(c)) - med if len(c) else float('nan')
        z = delta / noise
        HAND[(n, p)] = z
        cols.append(f'{delta:+9.0f} (z{z:+6.1f})')
    raw = phase_values(n, [s[0] for s in steps])
    n_bad = sum(is_glitch(v) for v in raw)
    print(f'{n:20s} {med:12.0f} {noise:7.0f} {cols[0]:>18s} {cols[1]:>18s}   {n_bad}/{len(raw)}')

print()
for n in NAMES:
    hits = [p.split('_')[1] for p in ('press_left', 'press_right') if abs(HAND[(n, p)]) >= RESPONSE_Z]
    own = 'left' if n.startswith('l_') else 'right' if n.startswith('r_') else None
    if not hits:
        print(f'{n}: no response to either press. Try the motor squeeze below; if that also shows nothing, '
              'the sensor or its wiring is the problem.')
    elif hits == [own]:
        print(f'{n}: responds to its own gripper. Sign of the response: '
              f'{"up" if HAND[(n, "press_" + own)] > 0 else "down"} when squeezed.')
    else:
        print(f'{n}: responds to pressing {" and ".join(hits)}: '
              + ('wired to the other side, or the press moved the whole robot.' if own not in hits else 'check for crosstalk.'))

## 3. Motor squeeze (optional)

For each side in `SQUEEZE_SIDES`, the gripper opens, then closes slowly over `CLOSE_S` seconds and holds for `HOLD_S`.
Put a **soft object** (sponge, foam, folded cloth) between the fingers first and keep your fingers clear.
Only the gripper motor is turned on. If it doesn't follow the commands, the cell says so and skips that side.
Interrupt the kernel to stop: the gripper reopens and goes limp.

In [ ]:
SQUEEZE = {}
for side in SQUEEZE_SIDES:
    g = GRIPPERS.get(side)
    if g is None:
        print(f'no {side} gripper found; skipping.')
        continue
    sign = 1 if side == 'left' else -1
    g_open, g_closed = GRIPPER_OPEN_L * sign, GRIPPER_CLOSED_L * sign
    print(f'\n=== {side} gripper: put a soft object between the fingers. Starting in 5 s ...')
    time.sleep(5)

    g.goal_position = g.present_position
    g.compliant = False
    time.sleep(0.3)
    try:
        start = g.present_position
        record(f'{side}_open', 1.5, banner=f'opening to {g_open} deg',
               each=lambda tau: setattr(g, 'goal_position', start + (g_open - start) * tau))
        moved = abs(g.present_position - start)
        if abs(g_open - start) > 5 and moved < 2:
            print(f'  !! the {side} gripper did not move (commanded {start:.1f} -> {g_open} deg, '
                  f'now {g.present_position:.1f}, compliant={g.compliant}). Motor off or not responding: skipping.')
            continue
        record(f'{side}_open_rest', 1.5, banner='holding open (reference level)')
        start = g.present_position
        record(f'{side}_closing', CLOSE_S, banner=f'closing slowly to {g_closed} deg',
               each=lambda tau: setattr(g, 'goal_position', start + (g_closed - start) * tau))
        record(f'{side}_hold', HOLD_S, banner='holding closed')
        err = g.goal_position - g.present_position
        print(f'  stopped at {g.present_position:.1f} deg (goal {g.goal_position:.1f}, gap {err:+.1f}: a large gap means '
              'the fingers were blocked by the object)')
    finally:
        g.goal_position = g_open
        time.sleep(1.5)
        g.compliant = True
        print(f'  {side} gripper reopened and set compliant.')

    for n in NAMES:
        ref = clean(phase_values(n, [f'{side}_open_rest'], skip_s=0.3))
        med, noise = robust_stats(ref)
        noise = max(noise, BASE[n][1])
        hold = clean(phase_values(n, [f'{side}_hold'], skip_s=0.5))
        delta = float(np.median(hold)) - med if len(hold) else float('nan')
        z = delta / noise
        SQUEEZE[(n, side)] = (delta, z)
        verdict = 'RESPONDS' if abs(z) >= RESPONSE_Z else 'no response'
        print(f'  {n:20s} open {med:10.0f}   closed-on-object shift {delta:+9.0f} (z{z:+6.1f})   {verdict}')

## 4. Plot and save

Dots are clean readings; red crosses at the top mark glitch samples. Shaded bands are the press and squeeze windows.

In [ ]:
import matplotlib.pyplot as plt

t = np.array([r[0] for r in LOG])
fig, axes = plt.subplots(len(NAMES) + 1, 1, figsize=(12, 2.6 * (len(NAMES) + 1)), sharex=True)


def shade(ax):
    spans = {}
    for r in LOG:
        spans.setdefault(r[1], [r[0], r[0]])[1] = r[0]
    for p, (a, b) in spans.items():
        if p.startswith('press') or p.endswith(('closing', 'hold')):
            ax.axvspan(a, b, color='tab:orange' if 'left' in p else 'tab:blue', alpha=0.15)
            ax.text(a, 1.0, p, transform=ax.get_xaxis_transform(), fontsize=7, va='bottom')


for ax, n in zip(axes, NAMES):
    raw = [r[2 + NAMES.index(n)] for r in LOG]
    bad = np.array([is_glitch(v) for v in raw])
    y = np.array([as_float(v) if not b else np.nan for v, b in zip(raw, bad)])
    ax.plot(t[~bad], y[~bad], '.', ms=3)
    if bad.any():
        ax.plot(t[bad], np.full(bad.sum(), np.nanmax(y) if np.isfinite(y).any() else 0), 'rx', ms=4,
                label=f'glitch ({bad.sum()})')
        ax.legend(loc='lower right', fontsize=7)
    ax.set_ylabel(f'{n}\n(raw)')
    shade(ax)

k = 2 + len(NAMES)
axes[-1].plot(t, [r[k] for r in LOG], label='left present')
axes[-1].plot(t, [r[k + 1] for r in LOG], label='right present')
axes[-1].set_ylabel('gripper (deg)')
axes[-1].set_xlabel('time (s)')
axes[-1].legend(fontsize=7)
shade(axes[-1])
plt.tight_layout()
plt.show()

with open(OUTPUT_CSV, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['t', 'phase', *NAMES, 'l_gripper_deg', 'r_gripper_deg', 'l_goal_deg', 'r_goal_deg',
                *[f'{n}_glitch' for n in NAMES]])
    for r in LOG:
        w.writerow([*r, *[int(is_glitch(r[2 + i])) for i in range(len(NAMES))]])
print(f'wrote {len(LOG)} rows to {OUTPUT_CSV}')